In [ ]:
input_file = 'datasets/preprocessed/flores_plus.jsonl'

In [ ]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import os
import json
import csv

print(f"Checking dataset at {input_file}...")
if not os.path.exists(input_file):
    print(f"Error: {input_file} not found.")
else:
    required_keys = {"text", "label"}
    labels_found = set()
    error_records = []
    
    line_num = 0
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            line_num += 1
            error_msg = None
            try:
                record = json.loads(line)
                if not required_keys.issubset(record.keys()):
                    error_msg = f"Missing required keys. Found: {list(record.keys())}"
                elif not record.get("label"):
                    error_msg = "Empty label field."
                elif not record.get("text") or not str(record.get("text")).strip():
                    error_msg = "Empty text field."
                else:
                    labels_found.add(record.get("label"))
                    
                if error_msg:
                    error_records.append({
                        "line_num": line_num,
                        "error": error_msg,
                        "raw_line": line.strip()
                    })
            except json.JSONDecodeError:
                error_records.append({
                    "line_num": line_num,
                    "error": "Invalid JSON.",
                    "raw_line": line.strip()
                })
                
    if len(error_records) == 0:
        print(f"Dataset check passed! Validated {line_num} records.")
        print(f"Found {len(labels_found)} unique labels, e.g., {list(labels_found)[:5]}")
    else:
        print(f"Dataset check failed with {len(error_records)} errors.")
        
        # Save errors to CSV
        output_csv = input_file.replace(".jsonl", "_errors.csv")
        with open(output_csv, "w", encoding="utf-8", newline="") as csvfile:
            fieldnames = ["line_num", "error", "raw_line"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for er in error_records:
                writer.writerow(er)
        print(f"Exported error details to {output_csv}")


## Baseline model cross-check (Sinhala-labeled rows)

Run the char n-gram + Logistic Regression baseline (`models/langid_model.pkl`, trained in `notebooks/LangID_1_Models.ipynb`) on every row labeled `sin` in this dataset. Rows the baseline does **not** predict as `sinhala` are surfaced below. These are candidates for mislabeled Pali/Sanskrit-in-Sinhala-script text (or genuinely hard/ambiguous examples) worth a manual look.

In [ ]:
import joblib
import pandas as pd

MODEL_DIR = "../models"
SINHALA_LABEL = "sin"
MODEL_LABEL_FOR_SINHALA = "sinhala"  # baseline was trained on full language names

if not os.path.exists(input_file):
    print(f"Skipping baseline model check: {input_file} not found.")
    mismatches = pd.DataFrame(columns=["text", "label", "source", "predicted_label"])
else:
    sinhala_records = []
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            if record.get("label") == SINHALA_LABEL:
                sinhala_records.append(record)

    if not sinhala_records:
        print(f"No '{SINHALA_LABEL}'-labeled records in {input_file}; skipping baseline model check.")
        mismatches = pd.DataFrame(columns=["text", "label", "source", "predicted_label"])
    else:
        vectorizer = joblib.load(os.path.join(MODEL_DIR, "langid_vectorizer.pkl"))
        clf = joblib.load(os.path.join(MODEL_DIR, "langid_model.pkl"))

        sinhala_df = pd.DataFrame(sinhala_records)
        X = vectorizer.transform(sinhala_df["text"])
        sinhala_df["predicted_label"] = clf.predict(X)

        mismatches = sinhala_df[sinhala_df["predicted_label"] != MODEL_LABEL_FOR_SINHALA].reset_index(drop=True)

        print(f"Baseline model check (char n-gram + LogReg) on '{SINHALA_LABEL}'-labeled rows in {input_file}:")
        print(f"  {len(sinhala_df)} rows labeled '{SINHALA_LABEL}', {len(mismatches)} not predicted as '{MODEL_LABEL_FOR_SINHALA}'")

        if not mismatches.empty:
            print(mismatches["predicted_label"].value_counts().to_string())

            checks_dir = "datasets/checks"
            os.makedirs(checks_dir, exist_ok=True)
            dataset_name = os.path.splitext(os.path.basename(input_file))[0]
            mismatches_path = os.path.join(checks_dir, f"{dataset_name}_sinhala_mismatches.csv")
            mismatches.to_csv(mismatches_path, index=False)
            print(f"Saved mismatches to {mismatches_path}")

mismatches